<div style="text-align:center;">
  <img src="https://github.com/LinkedEarth/Logos/blob/master/PaleoPAL/PaleoPal_rectangular_light.png?raw=true" width="500">
</div>

## PaleoPAL Evaluation: Notebook 4 

This notebook is part of a series of evaluation tests for the [PaleoPAL](linked.earth/paleopal) assistant. You have two hours to complete the assignment. 

The notebook is divided into the following sections:
1. Data Gathering (1 hour and 15min)
2. Analysis (35 min)
3. Visualization (10min)

If you cannot complete the assignment for each section in the time alloted, use the solution and move on to the next section. 

**Use VS Code to complete the assignment**. 

In [51]:
### Import libraries

import ast
import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

import pyleoclim as pyleo
import pyleotups as pt
from pylipd.lipd import LiPD

## Context
Evaluating the statistical significance of spectral peaks is a longstanding problem in paleoclimatology. Given the pervasive degree of autocorrelation, a common benchmark for such significance tests is to generate a null distribution from the power spectral densities of timeseries generated from an AR(1) model, whose parameters (φ = lag-1 autocorrelation, σ2 = variance of innovation) are fit to the timeseries of interest (e.g. GISP2). 

In Pyleoclim, the default method is to do this via the "method of moments (MoM)". This estimator is not optimal. However, back in 2022, Lionel Voirol (University of Geneva) also introduced a maximum-likelihood method, which works natively on unevenly-spaced data. The null distribution is then estimated via a parametric bootstrap approach. 

The question is: does it matter in practice how if significance bounds are derived optimally or not?

## Data Gathering (1hr and 15min)

In this part of the assignment, you will gather data from three different data sources: : a SPARQL endpoint to the LiPDGraph, PANGAEA through PyleoTUPS, and a local LiPD file using PyLiPD. 


### LiPDGraph

Using a SPARQL query on the LiPDGraph, retrieve all `Glacier ice` records from the `Temp12k` compilation and filter the results for a dataset containing the name `Huascaran`.

**Assignment Task:**

a. Query the LiPDGraph endpoint (https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic) for all records in the `Temp12k` compilation based on `Glacier ice`. Return information about the name of the dataset, the type of archive, geographical location, name of the variables, name of the compilation, values for time and paleo variable and associated units. 
b. Convert the query response into a `pandas.DataFrame`.
c. Filter the DataFrame for datasets containing the name `Huascaran`
d. Remove duplicate rows from the resulting DataFrame 

In [5]:
query = """

PREFIX le: <http://linked.earth/ontology#>
PREFIX le_var: <http://linked.earth/ontology/paleo_variables#>
PREFIX wgs84: <http://www.w3.org/2003/01/geo/wgs84_pos#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?dataSetName ?archiveType ?geo_meanLat ?geo_meanLon
    ?paleoData_variableName ?paleoData_TSID ?paleoData_values ?paleoData_units ?paleoData_proxy
    ?time_variableName ?time_TSID ?time_values ?time_units ?compilationName
WHERE {
    ?ds a le:Dataset .
    ?ds le:hasName ?dataSetName .

    ?ds le:hasArchiveType ?atObj . ?atObj rdfs:label ?archiveType .
    FILTER(regex(?archiveType, 'Glacier ice.*',"i"))


    ?ds le:hasLocation ?loc .
    OPTIONAL { ?loc wgs84:lat  ?geo_meanLat . }
    OPTIONAL { ?loc wgs84:long ?geo_meanLon . }

    ?ds le:hasPaleoData ?data .
    ?data le:hasMeasurementTable ?table .

    ?table le:hasVariable ?var .
    ?var le:partOfCompilation ?compilation . 
    ?compilation le:hasName ?compilationName .
    FILTER (?compilationName = "Temp12k").
    ?var le:hasName ?paleoData_variableName .
    ?var le:hasValues ?paleoData_values .
    OPTIONAL { ?var le:hasVariableId ?paleoData_TSID . }
    OPTIONAL { ?var le:hasUnits ?uObj . ?uObj rdfs:label ?paleoData_units . }
    OPTIONAL { ?var le:hasProxy ?pObj . ?pObj rdfs:label ?paleoData_proxy . }

    ?table le:hasVariable ?timevar .
    ?timevar le:hasName ?time_variableName .
    ?timevar le:hasValues ?time_values .
    OPTIONAL { ?timevar le:hasVariableId ?time_TSID . }
    OPTIONAL { ?timevar le:hasUnits ?tuObj . ?tuObj rdfs:label ?time_units . }
    ?timevar le:hasStandardVariable le_var:age .
}

""" 

url = 'https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic'

response = requests.post(url, data = {'query': query})

data = io.StringIO(response.text)
D_t12k = pd.read_csv(data, sep=",")

D_ice = D_t12k[D_t12k['dataSetName'].str.contains('Huascaran', case=False, na=False)]

D_ice = D_ice.iloc[0,:]

### PyleoTUPS

You will use the "Shackleton site" oxygen isotopes records. The collection of records from this site can be accessed through [PANGAEA](https://doi.pangaea.de/10.1594/PANGAEA.951401) and contains 9 datasets. Here, you will use the planktonic foraminifera *G. bulloides* $\delta^{18}O$, as they are more wiggly than the benthics and there might be more contentious peaks in there. 

**Assignment Task**
a. Search for the study using the information from the [PANGAEA page](https://doi.pangaea.de/10.1594/PANGAEA.951401).
b. Display the study summary so you can verify the citation and dataset identity.
c. Display the geographic metadata so you can recover the site name, latitude, and longitude.
d. Load the data table and inspect the first rows. Make note of the column names and how they can be used later.

In [31]:
dsp = pt.PangaeaDataset()
res = dsp.search_studies(study_ids='951386')
display(res)

df_geo = dsp.get_geo()
display(df_geo)

df_data = dsp.get_data(study_id = '951386')[0]
display(df_data.head())

### PyLiPD

Load the data in the `Botuvera.Brazil.2005.lpd` and retrieve information about the $\delta^{18}O$ timeseries.

**Assignment task:**
a. Open the dataset and retrieve relevant timeseries information
b. Filter the dataframe to keep only rows where the paleo variable is $\delta^{18}O$.


In [16]:
D = LiPD()
D.load('./Botuvera.Brazil.2005.lpd')

df = D.get_timeseries_essentials(D.get_all_dataset_names())

df_cut = df[df['paleoData_variableName'] == 'd18O']
df_cut

## Analysis (35 minutes)

You will create `pyleoclim.Series` objects from the data that you have gathered from the LiPDGraph, PANGAEA and your local LiPD file and perform spectral analysis using the Weighted Wavelet Z Transform. You will assess significant in two ways: using the MoM default and using the `uar1` method.

**Assignment task**

a. Create `pyleoclim.Series` objects from the records obtained in the previous section
b. Detrend the timeseries
c. Perform spectral analysis on each record using the Weighted Wavelet Z Transform.
d. Assess the significance using the default method and `uar1`. Set the number of simulations to 10 for computational efficiency.

<div style="background-color:#e6f2ff; border-left:5px solid #2f80ed; padding:12px 16px; margin-top:16px; border-radius:6px;">
  <strong>Note:</strong> A few things to keep in mind as you create your Series: a. LiPDGraph encodes data as a string (not an array), b. watch your units!
</div>

In [36]:
#load into Series
ts_botu = pyleo.Series(time=df_cut['time_values'].iloc[0], value = df_cut['paleoData_values'].iloc[0],
                  time_name = 'Age', time_unit = df_cut['time_units'].iloc[0],
                  value_name = "$\delta^{18}$O", 
                  value_unit = "‰ VPDB",
                  label = df_cut['dataSetName'].iloc[0],
                  archiveType = 'Speleothem', 
                  auto_time_params=False, verbose=False)
ts_botu = ts_botu.convert_time_unit('kyr BP')


t = np.asarray(ast.literal_eval(D_ice['time_values']), dtype=float)
v = np.asarray(ast.literal_eval(D_ice['paleoData_values']), dtype=float)

ts_huasc = pyleo.Series(
    time=t,
    value=v,
    time_name=D_ice['time_variableName'],
    time_unit=D_ice['time_units'],
    value_name=D_ice['paleoData_variableName'],
    value_unit=D_ice['paleoData_units'],
    label=D_ice['dataSetName'],
    archiveType='GlacierIce',
    auto_time_params=False, 
    verbose=False)

ts_shack = pyleo.Series(
    time=df_data['Age'],
    value=df_data['G. bulloides δ18O'],
    archiveType='MarineSediment',
    time_name='Age', time_unit='ka BP',
    label='U1385 G. bulloides',
    value_name='$\\delta^{18}$O', value_unit='‰',
    auto_time_params=False, verbose=False,
)

#Detrending
ts_botu = ts_botu.detrend()
ts_huasc = ts_huasc.detrend()
ts_shack = ts_shack.detrend()

# Spectral
nsim=10
psd_botu_ar1 = ts_botu.spectral(method = 'wwz').signif_test(method='ar1sim', number=nsim)
psd_botu_uar1 = ts_botu.spectral(method = 'wwz').signif_test(method='uar1', number=nsim)

psd_huasc_ar1 = ts_huasc.spectral(method = 'wwz').signif_test(method='ar1sim', number=nsim)
psd_huasc_uar1 = ts_huasc.spectral(method = 'wwz').signif_test(method='uar1', number=nsim)

psd_shack_ar1 = ts_shack.spectral(method = 'wwz').signif_test(method='ar1sim', number=nsim)
psd_shack_uar1 = ts_shack.spectral(method = 'wwz').signif_test(method='uar1', number=nsim)

## Visualization (10 min)

**Assignment task:**

Create a figure displaying the results between the two significance tests side-by-side for each of the record.


In [50]:
fig, axes = plt.subplots(3, 2, figsize=(12, 12), sharey=True)
psd_botu_ar1.plot(ax=axes[0, 0])
psd_botu_uar1.plot(ax=axes[0, 1])
axes[0, 0].set_title(f'{ts_botu.label} — ar1sim')
axes[0, 1].set_title(f'{ts_botu.label} — uar1')

psd_huasc_ar1.plot(ax=axes[1, 0])
psd_huasc_uar1.plot(ax=axes[1, 1])
axes[1, 0].set_title(f'{ts_huasc.label} — ar1sim')
axes[1, 1].set_title(f'{ts_huasc.label} — uar1')

psd_shack_ar1.plot(ax=axes[2, 0])
psd_shack_uar1.plot(ax=axes[2, 1])
axes[2, 0].set_title(f'{ts_shack.label} — ar1sim')
axes[2, 1].set_title(f'{ts_shack.label} — uar1')


plt.tight_layout()